# 03: Training SMILESTransformer Model with torch-molecule

This notebook trains a SMILESTransformerMolecularPredictor model using torch-molecule for toxicity prediction.

SMILESTransformer uses Transformer architectures on SMILES strings, leveraging attention mechanisms for sequence-based molecular classification.
This provides a different approach compared to GNN-based models (BFGNN, GRIN) that operate on graph structures.

**Note:** This model uses transformer-based sequence modeling on SMILES strings, treating molecules as sequences of tokens
rather than graph structures. This is complementary to the graph-based approaches.

Based on official torch-molecule documentation: https://liugangcode.github.io/torch-molecule/example.html

## Objectives

1. Load and prepare data for torch-molecule
2. Train torch-molecule SMILESTransformer model with hyperparameter optimization
3. Evaluate model performance
4. Compare with baseline MLP, BFGNN, and GRIN models



In [ ]:
# Setup
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, average_precision_score
import matplotlib.pyplot as plt

from src.data import load_clintox
from src.pipelines import (
    load_clintox_dataset,
    prepare_gnn_data,
    train_gnn_pipeline,
    evaluate_gnn_pipeline,
    save_gnn_model
)
from src.models import create_torch_molecule_model
from src.utils import set_seed, get_default_config, load_metrics

# Set seed for reproducibility
set_seed(42)
config = get_default_config()

print("✓ Imports successful")



## Load Data

Load the ClinTox dataset. torch-molecule models accept SMILES strings directly.



In [ ]:
# Load dataset using pipeline function
cache_dir = project_root / "data"
train_df, val_df, test_df = load_clintox_dataset(
    cache_dir=str(cache_dir),
    split_type="scaffold",
    seed=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Class distribution - Train: Toxic={train_df['CT_TOX'].sum()}, Non-toxic={len(train_df) - train_df['CT_TOX'].sum()}")

# Prepare data for SMILESTransformer model
X_train, y_train, X_val, y_val, X_test, y_test = prepare_gnn_data(train_df, val_df, test_df)

print(f"\n✓ Data prepared for torch-molecule SMILESTransformer")
print(f"  X_train: {len(X_train)} SMILES strings")
print(f"  X_val: {len(X_val)} SMILES strings")
print(f"  X_test: {len(X_test)} SMILES strings")



## Create SMILESTransformer Model

Initialize SMILESTransformerMolecularPredictor according to the official documentation.
For binary classification, we set `task_type='classification'` and `num_task=1`.

**Note:** SMILESTransformer uses Transformer architectures on SMILES strings, treating molecules as sequences.
This model learns attention-based representations over the SMILES token sequence, which can capture long-range
dependencies and patterns in molecular structures.



In [ ]:
# Import torch-molecule components
try:
    from torch_molecule import SMILESTransformerMolecularPredictor
    from torch_molecule.utils.search import ParameterType, ParameterSpec
    print("✓ torch-molecule SMILESTransformer imported successfully")
except ImportError as e:
    print(f"⚠ Error importing torch-molecule SMILESTransformer: {e}")
    print("Install with: pip install torch-molecule torch-geometric torch-scatter transformers")
    raise

# Initialize SMILESTransformer model with proper parameters for binary classification
# Based on documentation: https://liugangcode.github.io/torch-molecule/example.html
try:
    model = SMILESTransformerMolecularPredictor(
        num_task=1,  # Single binary classification task
        task_type="classification",  # Binary classification
        batch_size=32,  # Batch size for training
        epochs=50,  # Number of training epochs
        evaluate_criterion='roc_auc',  # Evaluation metric
        evaluate_higher_better=True,  # Higher ROC-AUC is better
        verbose='progress_bar'  # Show progress bar during training
    )
    print(f"✓ SMILESTransformerMolecularPredictor created successfully")
    print(f"  Model type: {type(model)}")
except Exception as e:
    print(f"⚠ Error creating SMILESTransformer model: {e}")
    print("Trying with minimal parameters...")
    try:
        # Fallback: try with fewer parameters
        model = SMILESTransformerMolecularPredictor(
            num_task=1,
            task_type="classification",
            batch_size=32,
            epochs=50,
            verbose='progress_bar'
        )
        print("✓ SMILESTransformer model created with minimal parameters")
    except Exception as e2:
        print(f"⚠ Error with minimal parameters: {e2}")
        import traceback
        traceback.print_exc()
        model = None
        print("\nNote: SMILESTransformer model may not be available in this torch-molecule version.")
        print("      Please check torch-molecule documentation or try a different model.")



In [ ]:
# Define hyperparameter search space for SMILESTransformer
# Based on documentation example and SMILESTransformer architecture
# SMILESTransformer uses transformer architecture with specific parameter names
try:
    from torch_molecule.utils.search import ParameterType, ParameterSpec
    
    search_parameters = {
        # Transformer architecture parameters
        "hidden_size": ParameterSpec(ParameterType.INTEGER, (128, 512)),
        "n_heads": ParameterSpec(ParameterType.INTEGER, (4, 16)),
        "num_layers": ParameterSpec(ParameterType.INTEGER, (2, 6)),
        "dim_feedforward": ParameterSpec(ParameterType.INTEGER, (256, 1024)),
        # Optimization parameters
        "learning_rate": ParameterSpec(ParameterType.LOG_FLOAT, (1e-5, 1e-2)),
        "weight_decay": ParameterSpec(ParameterType.LOG_FLOAT, (1e-10, 1e-3)),
        "dropout": ParameterSpec(ParameterType.FLOAT, (0.0, 0.5)),
        "scheduler_factor": ParameterSpec(ParameterType.FLOAT, (0.1, 0.9)),
    }
    
    print("✓ Hyperparameter search space defined for SMILESTransformer")
    print(f"  Number of hyperparameters: {len(search_parameters)}")
    print(f"  Valid parameters: {list(search_parameters.keys())}")
except ImportError:
    print("⚠ Could not import ParameterSpec, will skip hyperparameter search")
    search_parameters = None



## Train Model with Hyperparameter Optimization

Use `autofit()` method as shown in the official documentation.
This will automatically search for the best hyperparameters.

**Note:** SMILESTransformer processes SMILES strings as sequences using transformer architecture,
which can effectively capture patterns and dependencies in molecular representations.



In [ ]:
# Train SMILESTransformer model using pipeline function
if model is not None:
    print("=" * 70)
    print("Training SMILESTransformer model with hyperparameter optimization")
    print("=" * 70)
    print(f"Training data: {len(X_train)} samples")
    print(f"Validation data: {len(X_val)} samples")
    print(f"Class distribution - Toxic: {y_train.sum()}, Non-toxic: {len(y_train) - y_train.sum()}")
    print(f"Hyperparameter search trials: {config.get('torch_molecule', {}).get('n_trials', 20)}")
    print()
    
    # Convert labels to list of lists format
    y_train_list = [[int(y)] for y in y_train]
    y_val_list = [[int(y)] for y in y_val]
    
    try:
        # Use autofit() for hyperparameter optimization if available
        n_trials = config.get('torch_molecule', {}).get('n_trials', 20)
        
        if hasattr(model, 'autofit') and search_parameters:
            model.autofit(
                X_train=X_train,  # List of SMILES strings
                y_train=y_train_list,  # List of lists: [[0], [1], ...]
                X_val=X_val,
                y_val=y_val_list,
                n_trials=n_trials,
                search_parameters=search_parameters
            )
            print("\n✓ SMILESTransformer model training completed with hyperparameter optimization!")
        else:
            # Fallback to regular fit() if autofit() not available
            print("⚠ autofit() not available or search_parameters not defined, using fit()...")
            model.fit(
                X_train=X_train,
                y_train=y_train_list,
                X_val=X_val,
                y_val=y_val_list
            )
            print("\n✓ SMILESTransformer model training completed with fit()!")
        
    except Exception as e:
        print(f"\n⚠ Error during training: {e}")
        import traceback
        traceback.print_exc()
        raise
else:
    print("⚠ SMILESTransformer model not available for training")



In [ ]:
# Evaluate on validation set
if model is not None:
    print("=" * 70)
    print("Validation Set Evaluation (SMILESTransformer)")
    print("=" * 70)
    
    try:
        val_metrics = evaluate_gnn_pipeline(model, val_df)
        
        print(f"\nValidation Metrics:")
        print(f"  AUC-ROC: {val_metrics['auc_roc']:.4f}")
        print(f"  Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"  F1 Score: {val_metrics['f1']:.4f}")
        print(f"  PR-AUC: {val_metrics['pr_auc']:.4f}")
        print(f"  AUPRC: {val_metrics['auprc']:.4f}")
        
    except Exception as e:
        print(f"⚠ Error during validation: {e}")
        import traceback
        traceback.print_exc()
        val_metrics = None
else:
    val_metrics = None



## Evaluate on Test Set

Evaluate the trained HFPretrained model on the test set.
This includes AUPRC as a metric for imbalanced classification.



In [ ]:
# Evaluate on test set using pipeline function
if model is not None:
    print("=" * 70)
    print("Test Set Evaluation (SMILESTransformer)")
    print("=" * 70)
    
    try:
        test_metrics = evaluate_gnn_pipeline(model, test_df)
        
        print(f"\nTest Set Metrics:")
        print(f"  AUC-ROC: {test_metrics['auc_roc']:.4f}")
        print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
        print(f"  F1 Score: {test_metrics['f1']:.4f}")
        print(f"  PR-AUC: {test_metrics['pr_auc']:.4f}")
        print(f"  AUPRC: {test_metrics['auprc']:.4f}")
        
    except Exception as e:
        print(f"⚠ Error during test evaluation: {e}")
        import traceback
        traceback.print_exc()
        test_metrics = None
else:
    test_metrics = None



## Compare with Baseline, BFGNN, and GRIN Models

Compare SMILESTransformer model performance with all other trained models.



In [ ]:
# Load baseline, BFGNN, and GRIN metrics if available
models_dir = project_root / "models"
comparison_data = []

# Load Baseline MLP metrics
baseline_metrics_path = models_dir / "baseline_metrics.txt"
if baseline_metrics_path.exists():
    baseline_metrics = load_metrics(str(baseline_metrics_path))
    baseline_auprc = baseline_metrics.get('auprc', baseline_metrics.get('pr_auc', 'N/A'))
    comparison_data.append({
        'Model': 'Baseline MLP (Fingerprint)',
        'AUC-ROC': baseline_metrics.get('auc_roc', 'N/A'),
        'Accuracy': baseline_metrics.get('accuracy', 'N/A'),
        'F1': baseline_metrics.get('f1', 'N/A'),
        'AUPRC': baseline_auprc
    })

# Load BFGNN metrics
bfgnn_metrics_path = models_dir / "torch_molecule_metrics.txt"
if bfgnn_metrics_path.exists():
    bfgnn_metrics = load_metrics(str(bfgnn_metrics_path))
    bfgnn_auprc = bfgnn_metrics.get('auprc', bfgnn_metrics.get('pr_auc', 'N/A'))
    comparison_data.append({
        'Model': 'BFGNN (torch-molecule)',
        'AUC-ROC': bfgnn_metrics.get('auc_roc', 'N/A'),
        'Accuracy': bfgnn_metrics.get('accuracy', 'N/A'),
        'F1': bfgnn_metrics.get('f1', 'N/A'),
        'AUPRC': bfgnn_auprc
    })

# Load GRIN metrics
grin_metrics_path = models_dir / "grin_model_metrics.txt"
if grin_metrics_path.exists():
    grin_metrics = load_metrics(str(grin_metrics_path))
    grin_auprc = grin_metrics.get('auprc', grin_metrics.get('pr_auc', 'N/A'))
    comparison_data.append({
        'Model': 'GRIN (torch-molecule)',
        'AUC-ROC': grin_metrics.get('auc_roc', 'N/A'),
        'Accuracy': grin_metrics.get('accuracy', 'N/A'),
        'F1': grin_metrics.get('f1', 'N/A'),
        'AUPRC': grin_auprc
    })

# Add SMILESTransformer metrics
if test_metrics is not None:
    smilestransformer_auprc = test_metrics.get('auprc', test_metrics.get('pr_auc', 'N/A'))
    comparison_data.append({
        'Model': 'SMILESTransformer (torch-molecule)',
        'AUC-ROC': test_metrics.get('auc_roc', 'N/A'),
        'Accuracy': test_metrics.get('accuracy', 'N/A'),
        'F1': test_metrics.get('f1', 'N/A'),
        'AUPRC': smilestransformer_auprc
    })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("\n" + "=" * 70)
    print("Model Comparison: Baseline MLP vs BFGNN vs GRIN vs SMILESTransformer")
    print("=" * 70)
    print(comparison_df.to_string(index=False))
    
    # Visualize comparison
    if len(comparison_data) > 1:
        # Create subplots based on number of metrics
        metrics_to_plot = ['AUC-ROC', 'Accuracy', 'F1', 'AUPRC']
        available_metrics = [
            m for m in metrics_to_plot 
            if any(row.get(m, 'N/A') != 'N/A' for row in comparison_data)
        ]
        
        n_metrics = len(available_metrics)
        if n_metrics <= 4:
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        else:
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, metric in enumerate(available_metrics):
            values = []
            labels = []
            for row in comparison_data:
                val = row[metric]
                if val != 'N/A' and isinstance(val, (int, float)):
                    values.append(float(val))
                    labels.append(row['Model'])
            
            if values:
                colors = ['skyblue', 'salmon', 'lightgreen', 'gold'][:len(values)]
                axes[idx].bar(labels, values, alpha=0.7, color=colors)
                axes[idx].set_ylabel(metric)
                axes[idx].set_title(f'{metric} Comparison')
                axes[idx].grid(axis='y', alpha=0.3)
                axes[idx].set_ylim([0, 1])
                axes[idx].tick_params(axis='x', rotation=45)
        
        # Hide unused subplots
        for idx in range(len(available_metrics), len(axes)):
            axes[idx].axis('off')
        
        plt.suptitle('Model Performance Comparison: MLP vs BFGNN vs GRIN vs SMILESTransformer', 
                     fontsize=16, fontweight='bold', y=0.995)
        plt.tight_layout()
        
        # Save figure
        figures_dir = project_root / "output" / "figures"
        figures_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(figures_dir / "03_smilestransformer_model_comparison.png", dpi=300, bbox_inches='tight')
        print(f"\n✓ Comparison figure saved to: {figures_dir / '03_smilestransformer_model_comparison.png'}")
        
        plt.show()
else:
    print("No comparison data available")



In [ ]:
# Save model and metrics using pipeline function
if model is not None and test_metrics is not None:
    models_dir = project_root / "models"
    models_dir.mkdir(exist_ok=True)
    
    model_path, metrics_path = save_gnn_model(
        model=model,
        metrics=test_metrics,
        model_dir=models_dir,
        model_name="smilestransformer_model"
    )
    
    print(f"✓ SMILESTransformer model saved to: {model_path}")
    print(f"✓ Metrics saved to: {metrics_path}")
else:
    print("Model or metrics not available for saving")



## Summary

✓ SMILESTransformer model trained with hyperparameter optimization  
✓ Model evaluated on validation and test sets  
✓ Comparison with baseline MLP, BFGNN, and GRIN completed  

**Key Findings:**
- SMILESTransformer uses Transformer architecture on SMILES strings, treating molecules as sequences
- Leverages attention mechanisms to capture long-range dependencies and patterns in molecular representations
- Provides a sequence-based approach compared to GNN-based models (graph structure) and fingerprint-based models
- Performance comparison shows how different architectures (fingerprint, GNN, transformer) handle toxicity prediction

**Next Steps:**
- Proceed to `04_explainability_and_visualization.ipynb` to generate explanations for SMILESTransformer model
- Update `05_results_and_error_analysis.ipynb` to include SMILESTransformer in comprehensive analysis

**References:**
- torch-molecule documentation: https://liugangcode.github.io/torch-molecule/example.html
- torch-molecule GitHub: https://github.com/liugangcode/torch-molecule
- Model Overview: https://liugangcode.github.io/torch-molecule/overview.html
- Transformer Architecture: Attention mechanism for sequence modeling

